# Batter Hits — Hierarchical Bayesian Model (Mockup)

**Goal**: For each player today, generate a posterior predictive distribution over hits and compare to bookmaker lines to find edges.

**Model**:
- Each player has a `hit_rate` (probability of getting a hit per AB)
- `hit_rate` is drawn from a league-wide prior — so players with less history borrow strength from the league
- We observe their historical hits + ABs across seasons
- We predict today's hits given today's expected ABs

In [ ]:
import duckdb
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt

conn = duckdb.connect('../data/mlb.db')

## 1. Pull historical data

In [ ]:
# Pull players who are in today's prop lines
batting = conn.execute('''
    SELECT
        b.player_name,
        b.season,
        b.h,
        b.ab
    FROM stg_batting_stats b
    WHERE b.player_name IN (
        SELECT DISTINCT player_name FROM prop_lines WHERE market = 'batter_hits'
    )
    AND b.ab > 0
    ORDER BY b.player_name, b.season
''').df()

print(f"{batting['player_name'].nunique()} players matched to today's lines")
batting.head(10)

## 2. Prep for modeling

In [ ]:
# Encode players as integer indices
players = batting['player_name'].unique()
player_idx = {p: i for i, p in enumerate(players)}
n_players = len(players)

idx = batting['player_name'].map(player_idx).values
hits = batting['h'].values
ab = batting['ab'].values

coords = {
    'player': players,
    'obs': np.arange(len(batting))
}

print(f"Players: {n_players} | Observations: {len(batting)}")

## 3. Build the model

In [ ]:
with pm.Model(coords=coords) as hits_model:

    # --- League-wide hyperpriors ---
    # The average MLB hit rate is ~.250, so we center the prior there
    mu_alpha = pm.Normal('mu_alpha', mu=0, sigma=1)       # league mean (logit scale)
    sigma_alpha = pm.HalfNormal('sigma_alpha', sigma=0.5) # player-to-player variance

    # --- Per-player hit rate (non-centered parameterization) ---
    alpha_offset = pm.Normal('alpha_offset', 0, 1, dims='player')
    alpha = pm.Deterministic(
        'alpha',
        mu_alpha + sigma_alpha * alpha_offset,
        dims='player'
    )
    hit_rate = pm.Deterministic(
        'hit_rate',
        pm.math.sigmoid(alpha),
        dims='player'
    )

    # --- Likelihood ---
    # hits ~ Binomial(n=AB, p=hit_rate)
    h_obs = pm.Binomial(
        'h_obs',
        n=ab,
        p=hit_rate[idx],
        observed=hits,
        dims='obs'
    )

print(pm.model_to_graphviz(hits_model))

## 4. Prior predictive check

In [ ]:
with hits_model:
    prior_pred = pm.sample_prior_predictive(draws=500, random_seed=42)

# Does our prior produce plausible batting averages?
prior_rates = pm.math.sigmoid(prior_pred.prior['alpha'].values).eval()
print(f"Prior hit rate range: [{prior_rates.min():.3f}, {prior_rates.max():.3f}]")
print(f"Prior hit rate mean: {prior_rates.mean():.3f}")

## 5. Sample the posterior

In [ ]:
with hits_model:
    idata = pm.sample(
        draws=1000,
        tune=1000,
        chains=4,
        nuts_sampler='nutpie',
        random_seed=42,
    )

idata.to_netcdf('../data/hits_model_idata.nc')
print("Saved.")

## 6. Diagnostics

In [ ]:
n_div = idata.sample_stats['diverging'].sum().item()
print(f"Divergences: {n_div}")

summary = az.summary(idata, var_names=['mu_alpha', 'sigma_alpha'])
print(summary[['mean', 'sd', 'hdi_3%', 'hdi_97%', 'ess_bulk', 'r_hat']])

## 7. Predict today's hits & compare to lines

In [ ]:
# Pull today's lines (DraftKings, Over only)
lines = conn.execute('''
    SELECT player_name, line, odds
    FROM prop_lines
    WHERE market = 'batter_hits'
    AND bookmaker = 'draftkings'
    AND side = 'Over'
    AND player_name IN (SELECT DISTINCT player_name FROM stg_batting_stats)
''').df()

# Typical ABs per game (~3.5 for most starters)
expected_ab = 4

results = []
hit_rate_samples = idata.posterior['hit_rate'].values  # shape: (chains, draws, players)
hit_rate_flat = hit_rate_samples.reshape(-1, n_players)  # (samples, players)

for _, row in lines.iterrows():
    name = row['player_name']
    line = row['line']
    odds = row['odds']

    if name not in player_idx:
        continue

    pidx = player_idx[name]
    rates = hit_rate_flat[:, pidx]

    # Simulate hits for today given expected ABs
    sim_hits = np.random.binomial(n=expected_ab, p=rates)

    # P(hits > line) from our model
    model_prob_over = (sim_hits > line).mean()

    # Implied probability from American odds
    if odds < 0:
        implied_prob = (-odds) / (-odds + 100)
    else:
        implied_prob = 100 / (odds + 100)

    edge = model_prob_over - implied_prob

    results.append({
        'player': name,
        'line': line,
        'odds': odds,
        'model_prob_over': round(model_prob_over, 3),
        'implied_prob': round(implied_prob, 3),
        'edge': round(edge, 3),
    })

results_df = pd.DataFrame(results).sort_values('edge', ascending=False)
print("Top edges (model prob > implied prob):")
results_df.head(15)

## 8. Visualize a player's predictive distribution

In [ ]:
# Pick a player from today's lines
player_name = results_df.iloc[0]['player']
line = results_df.iloc[0]['line']
pidx = player_idx[player_name]

rates = hit_rate_flat[:, pidx]
sim_hits = np.random.binomial(n=expected_ab, p=rates)

plt.figure(figsize=(8, 4))
plt.hist(sim_hits, bins=range(0, expected_ab + 2), align='left',
         rwidth=0.8, color='steelblue', edgecolor='white')
plt.axvline(line + 0.5, color='red', linestyle='--', label=f'Line: {line}')
plt.title(f'{player_name} — Predicted Hits Today')
plt.xlabel('Hits')
plt.ylabel('Samples')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Model P(over {line}): {(sim_hits > line).mean():.1%}")